In [2]:
import warnings
warnings.filterwarnings('ignore')

import os
import pickle

import numpy as np
import pandas as pd

from datetime import datetime, timedelta
from dateutil.relativedelta import relativedelta

from pandas.tseries.offsets import MonthEnd, MonthBegin
from maricovault.MaricoDB import MaricoSnowflake

from joblib import Parallel, delayed

In [3]:
def get_dbconnection(db_name): 

    KEY_VAULT_NAME = "prod-pwd"
    if db_name == 'PROD':
        db_name = 'prod'
    else:
        db_name = 'dev'

    msf = MaricoSnowflake(KEY_VAULT_NAME)
    msf.get_db_credentials(db_name=db_name)
    msf.connect()
    dbconnection = msf.get_connection()
    
    return dbconnection

In [4]:
dev_conn = get_dbconnection('DEV')
prod_conn = get_dbconnection('PROD')


Credentials retrieved successfully for dev db.

Credentials retrieved successfully for prod db.


### Helper Functions

In [5]:
realignment_df = pd.read_sql(
    """select * from trn_mil_asm_psku_realignment""",
    dev_conn
)
realignment_df.columns = realignment_df.columns.str.lower()

def demand_driver_realign_pskus(data, channel):
    """
    Realign the old pskus to new pskus and return updated data.

    Args:
        data: pandas dataframe
        - master dataframe having all the pskus
    
    Return:
        data: pandas dataframe
        - dataframe 
    """
    realignment_data = realignment_df.copy()
    realignment_data.columns = realignment_data.columns.str.lower()
    realignment_data = realignment_data[
        (realignment_data["channel"] == channel)
        | (realignment_data["channel"] == channel + " B2C")
        | (realignment_data["channel"] == "ALL")
    ]

    data["parent_material_code"] = data["parent_material_code"].astype(int)

    for grp, grp_data in realignment_data.groupby(by=["psku old", "asm"]):
        old_psku, old_asm = grp
        new_psku = grp_data["psku new"].values[0]
        if old_asm != "ALL":
            condition = (data["parent_material_code"] == old_psku) & (
                data["asm_area_code"] == old_asm
            )
        else:
            condition = data["parent_material_code"] == old_psku

        data.loc[condition, "parent_material_code"] = new_psku

    return data

In [8]:
x = realignment_df[realignment_df['psku new']==732460]

In [9]:
x

,psku realignment master,parent material_code old,asm,channel,new parent_material code,deletion indicator?,psku old,psku new
78,718431_ALL_GT,718431_PAR ADV JASMINE 45ml BTL,ALL,GT,731276_PAR ADV JASMINE GOLD 45ML BOT,None,718431,732460
139,731276_ALL_GT,731276_PAR ADV JASMINE GOLD 45ML BOT,ALL,GT,732460_PAR ADV JASMINE GOLD 45ML BOT NEW,None,731276,732460
221,718431_ALL_GT,None,ALL,GT,None,None,718431,732460
280,731276_ALL_GT,None,ALL,GT,None,None,731276,732460


### Forecast

### Actuals

In [10]:
actuals_query = """
SELECT
    CM.channel_name,
    CM.asm_area_code,
    CM.depot_code,
    MM.parent_material_code,
    MM.material_group_code,
    LAST_DAY(MESR.month_date) AS month_date,
    SUM(pri_actuals_vol_rum) AS pri_actuals_vol_rum,
    SUM(pri_apo_plan_vol_rum) AS pri_apo_plan_vol_rum,
    SUM(MESR.sec_apo_plan_vol_rum) AS sec_apo_plan_vol_rum,
    SUM(MESR.sec_actuals_vol_rum) AS sec_actuals_vol_rum
FROM
    dwh_bpm_dist_sku_daily MESR
JOIN
(
    SELECT
        material_code,
        parent_material_code,
        material_group_code,
        uom_reporting,
        vol_per_unit
    FROM 
        mst_material
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) MM ON MESR.material_code = MM.material_code
JOIN
(
    SELECT DISTINCT
        channel_name, 
        asm_area_code,
        customer_code,
        depot_code
    FROM
        mst_customer
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
) CM ON MESR.distributor_code = CM.customer_code
WHERE
    month_date BETWEEN '2026-01-01' AND '2026-03-31' AND
    CM.channel_name IN ('MT', 'E-Commerce', 'Q-Commerce', 'GT')
GROUP BY 1, 2, 3, 4, 5, 6
ORDER BY 1, 2, 3, 4, 6
"""

actuals_df = pd.read_sql(
    actuals_query,
    prod_conn
)

In [11]:
actuals_df.columns = actuals_df.columns.str.lower()
actuals_df['month_date'] = pd.to_datetime(actuals_df['month_date'])

In [12]:
actuals_df.head()

,channel_name,asm_area_code,depot_code,parent_material_code,material_group_code,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum
0,E-Commerce,BCE1,D231,718303,REV.LQDST,2026-01-31,0.0,0.00495,0.0,0.0
1,E-Commerce,BCE1,D231,718303,REV.LQDST,2026-02-28,0.0,0.00345,0.0,0.0
2,E-Commerce,BCE1,D231,718303,REV.LQDST,2026-03-31,0.0,0.00355,0.0,0.0
3,E-Commerce,BCE1,D231,718312,PCNO(R),2026-01-31,0.0,0.00495,0.0,0.0
4,E-Commerce,BCE1,D231,718312,PCNO(R),2026-02-28,0.0,0.00705,0.0,0.0


In [13]:
actuals_df.duplicated(subset=['channel_name', 'asm_area_code', 'depot_code', 'parent_material_code','month_date']).sum()

0

In [14]:
actuals_df['channel_name'] = actuals_df['channel_name'].replace({
    'E-Commerce': 'ECOM', 'Q-Commerce': 'QCOM'})

In [15]:
actuals_df

,channel_name,asm_area_code,depot_code,parent_material_code,material_group_code,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum
0,ECOM,BCE1,D231,718303,REV.LQDST,2026-01-31,0.0,0.004950,0.0,0.0
1,ECOM,BCE1,D231,718303,REV.LQDST,2026-02-28,0.0,0.003450,0.0,0.0
2,ECOM,BCE1,D231,718303,REV.LQDST,2026-03-31,0.0,0.003550,0.0,0.0
3,ECOM,BCE1,D231,718312,PCNO(R),2026-01-31,0.0,0.004950,0.0,0.0
4,ECOM,BCE1,D231,718312,PCNO(R),2026-02-28,0.0,0.007050,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...
123540,QCOM,QCW2,D463,810674,PA_ESS_HO,2026-03-31,0.0,0.147826,0.0,0.0
123541,QCOM,QCW2,D463,810971,PA_ESS_HO,2026-01-31,0.0,0.454555,0.0,0.0
123542,QCOM,QCW2,D463,810971,PA_ESS_HO,2026-02-28,0.0,0.454555,0.0,0.0
123543,QCOM,QCW2,D463,810971,PA_ESS_HO,2026-03-31,0.0,0.434782,0.0,0.0


In [16]:
import numpy as np

actuals_df['final_channel'] = np.where(
    actuals_df['channel_name'].isin(['QCOM', 'GT']),
    actuals_df['channel_name'],
    np.where(
        actuals_df['channel_name'].isin(['ECOM', 'MT']) &
        actuals_df['asm_area_code'].astype(str).str.startswith('B'),
        'B2B',
        actuals_df['channel_name']
    )
)
actuals_df

,channel_name,asm_area_code,depot_code,parent_material_code,material_group_code,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum,final_channel
0,ECOM,BCE1,D231,718303,REV.LQDST,2026-01-31,0.0,0.004950,0.0,0.0,B2B
1,ECOM,BCE1,D231,718303,REV.LQDST,2026-02-28,0.0,0.003450,0.0,0.0,B2B
2,ECOM,BCE1,D231,718303,REV.LQDST,2026-03-31,0.0,0.003550,0.0,0.0,B2B
3,ECOM,BCE1,D231,718312,PCNO(R),2026-01-31,0.0,0.004950,0.0,0.0,B2B
4,ECOM,BCE1,D231,718312,PCNO(R),2026-02-28,0.0,0.007050,0.0,0.0,B2B
...,...,...,...,...,...,...,...,...,...,...,...
123540,QCOM,QCW2,D463,810674,PA_ESS_HO,2026-03-31,0.0,0.147826,0.0,0.0,QCOM
123541,QCOM,QCW2,D463,810971,PA_ESS_HO,2026-01-31,0.0,0.454555,0.0,0.0,QCOM
123542,QCOM,QCW2,D463,810971,PA_ESS_HO,2026-02-28,0.0,0.454555,0.0,0.0,QCOM
123543,QCOM,QCW2,D463,810971,PA_ESS_HO,2026-03-31,0.0,0.434782,0.0,0.0,QCOM


In [17]:
actuals_df = actuals_df.groupby(['final_channel', 'asm_area_code', 'depot_code', 'parent_material_code',
       'material_group_code', 'month_date'])[['pri_actuals_vol_rum',
       'pri_apo_plan_vol_rum', 'sec_apo_plan_vol_rum', 'sec_actuals_vol_rum']].sum().reset_index()
actuals_df

,final_channel,asm_area_code,depot_code,parent_material_code,material_group_code,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum
0,B2B,BCE1,D231,718303,REV.LQDST,2026-01-31,0.000,0.007425,0.0100,0.000
1,B2B,BCE1,D231,718303,REV.LQDST,2026-02-28,0.000,0.004600,0.0070,0.000
2,B2B,BCE1,D231,718303,REV.LQDST,2026-03-31,0.014,0.005325,0.0071,0.014
3,B2B,BCE1,D231,718312,PCNO(R),2026-01-31,0.040,0.007425,0.0100,0.040
4,B2B,BCE1,D231,718312,PCNO(R),2026-02-28,0.080,0.009400,0.0140,0.080
...,...,...,...,...,...,...,...,...,...,...
119340,QCOM,QCW2,D463,810674,PA_ESS_HO,2026-03-31,0.000,0.147826,0.0000,0.000
119341,QCOM,QCW2,D463,810971,PA_ESS_HO,2026-01-31,0.000,0.454555,0.0000,0.000
119342,QCOM,QCW2,D463,810971,PA_ESS_HO,2026-02-28,0.000,0.454555,0.0000,0.000
119343,QCOM,QCW2,D463,810971,PA_ESS_HO,2026-03-31,0.000,0.434782,0.0000,0.000


In [18]:
actuals_df.rename(columns = {'final_channel':'channel_name'}, inplace = True)

In [19]:
vol_cols = [
    'pri_actuals_vol_rum',
    'pri_apo_plan_vol_rum',
    'sec_apo_plan_vol_rum',
    'sec_actuals_vol_rum'
]

actuals_df[vol_cols] = actuals_df[vol_cols].clip(lower=0)

In [20]:
tmp_df = pd.DataFrame()

for c in ['GT', 'MT', 'ECOM', 'QCOM','B2B']:
    tmp2_df = actuals_df[actuals_df['channel_name'] == c]
    tmp2_df = demand_driver_realign_pskus(tmp2_df, channel=c)
    tmp_df = pd.concat([tmp_df, tmp2_df], ignore_index=True)

In [21]:
actuals_df.duplicated(subset=['channel_name', 'asm_area_code', 'depot_code', 'parent_material_code','month_date']).sum()

0

In [22]:
tmp_df['month_date'].unique()

<DatetimeArray>
['2026-01-31 00:00:00', '2026-02-28 00:00:00', '2026-03-31 00:00:00']
Length: 3, dtype: datetime64[ns]

In [23]:
tmp_df = tmp_df.groupby(
    ['channel_name', 'asm_area_code', 'depot_code', 
     'parent_material_code', 'month_date'], as_index=False
).sum()

In [24]:
actuals_df = tmp_df.copy()

del tmp_df, tmp2_df

In [25]:
actuals_df#.head()

,channel_name,asm_area_code,depot_code,parent_material_code,month_date,material_group_code,pri_actuals_vol_rum,pri_apo_plan_vol_rum,sec_apo_plan_vol_rum,sec_actuals_vol_rum
0,B2B,BCE1,D231,718303,2026-01-31,REV.LQDST,0.000,0.007425,0.0100,0.000
1,B2B,BCE1,D231,718303,2026-02-28,REV.LQDST,0.000,0.004600,0.0070,0.000
2,B2B,BCE1,D231,718303,2026-03-31,REV.LQDST,0.014,0.005325,0.0071,0.014
3,B2B,BCE1,D231,718312,2026-01-31,PCNO(R),0.040,0.007425,0.0100,0.040
4,B2B,BCE1,D231,718312,2026-02-28,PCNO(R),0.080,0.009400,0.0140,0.080
...,...,...,...,...,...,...,...,...,...,...
116552,QCOM,QCW2,D463,810674,2026-03-31,PA_ESS_HO,0.000,0.147826,0.0000,0.000
116553,QCOM,QCW2,D463,810971,2026-01-31,PA_ESS_HO,0.000,0.454555,0.0000,0.000
116554,QCOM,QCW2,D463,810971,2026-02-28,PA_ESS_HO,0.000,0.454555,0.0000,0.000
116555,QCOM,QCW2,D463,810971,2026-03-31,PA_ESS_HO,0.000,0.434782,0.0000,0.000


In [26]:
query = """SELECT
        parent_material_code,
        material_group_code,
        uom_reporting,
        vol_per_unit
    FROM 
        mst_material
    WHERE
        company_code='MIL' AND
        latest_record_ind=1
        """
material_df = pd.read_sql(
    query,
    prod_conn
)

In [27]:
material_df.columns = material_df.columns.str.lower()
material_df['parent_material_code'] = material_df['parent_material_code'].astype(int)

In [28]:
actuals_df.drop(['material_group_code'], axis=1, inplace=True)

In [29]:
material_df[['parent_material_code', 'material_group_code']].drop_duplicates()

,parent_material_code,material_group_code
0,706643,SW RSHGEL
1,706644,SW RSHGEL
2,706316,SW RSHGEL
3,706317,SW RSHGEL
4,802817,SW GOLDEO
...,...,...
79512,705486,NHR ALM-R
79513,705487,NHR ALM-R
79514,705488,NHR ALM-R
79516,705202,SWK SFO


In [30]:
material_df[material_df['parent_material_code'] == 718480]

,parent_material_code,material_group_code,uom_reporting,vol_per_unit
547,718480,PA-BDYLOT,L,100.0
669,718480,PA-BDYLOT,L,100.0
769,718480,PA-BDYLOT,L,100.0
865,718480,PA-BDYLOT,L,100.0
978,718480,PA-BDYLOT,L,100.0
1021,718480,PA-BDYLOT,L,100.0
1388,718480,PA-BDYLOT,L,100.0
1556,718480,PA-BDYLOT,L,100.0
2084,718480,PA-BDYLOT,L,100.0
2085,718480,PA-BDYLOT,L,100.0


In [31]:
material_df = material_df[~((material_df['parent_material_code'] == 718480) & (material_df['material_group_code'] == 'P-ADVANCE'))]


In [32]:
actuals_df = actuals_df.merge(material_df[['parent_material_code', 'material_group_code']].drop_duplicates(), on='parent_material_code', how='left')

In [33]:
actuals_df = actuals_df.rename(columns={
    'channel_name': 'Channel',
    'asm_area_code': 'ASM',
    'depot_code': 'Depot',
    'parent_material_code': 'PSKU',
    'sec_apo_plan_vol_rum': 'Consensus Vol',
    'sec_actuals_vol_rum': 'Actuals Vol'
})

In [34]:
def read_qtr_ind_rate_table():
    query = """select * from DWH_SAP_INDEX_TURNOVER_MONTHWISE 
                where latest_rate_flag=1 and company_code='MIL'"""
    qtr_ind_rate_data = pd.read_sql(con=prod_conn, sql=query)
    qtr_ind_rate_data.columns = qtr_ind_rate_data.columns.str.lower()
    qtr_ind_rate =  qtr_ind_rate_data[['date', 'brand_code', 'turnover']]
    qtr_ind_rate = qtr_ind_rate.rename(columns= {'date':'month_date', 'turnover':'qtr_ind_rate'})
    
    return qtr_ind_rate

qtr_ind_rate_df = read_qtr_ind_rate_table()
qtr_ind_rate_df.head()

,month_date,brand_code,qtr_ind_rate
0,2027-03-31,NHR_NHO_E,266.579530
1,2027-03-31,SAFF SALT,38873.867804
2,2027-03-31,CO_SO_FS,2185.829959
3,2027-03-31,CO_SO_COC,0.000000
4,2027-03-31,SAF-MUSLI,315513.490535


In [35]:
actuals_df.rename(columns={'material_group_code': 'brand'}, inplace=True)

In [36]:
df = actuals_df.copy()
len_before_merge = len(df)
df = df.merge(
    qtr_ind_rate_df.drop('month_date', axis=1).rename(
        columns={'brand_code': 'brand', 'qtr_ind_rate': 'Index Rate'}
    ),
    on=['brand'], 
    how='left'
)
assert len_before_merge == len(df)
del len_before_merge

In [37]:
df

,Channel,ASM,Depot,PSKU,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,Consensus Vol,Actuals Vol,brand,Index Rate
0,B2B,BCE1,D231,718303,2026-01-31,0.000,0.007425,0.0100,0.000,REV.LQDST,247926.608903
1,B2B,BCE1,D231,718303,2026-02-28,0.000,0.004600,0.0070,0.000,REV.LQDST,247926.608903
2,B2B,BCE1,D231,718303,2026-03-31,0.014,0.005325,0.0071,0.014,REV.LQDST,247926.608903
3,B2B,BCE1,D231,718312,2026-01-31,0.040,0.007425,0.0100,0.040,PCNO(R),349274.001420
4,B2B,BCE1,D231,718312,2026-02-28,0.080,0.009400,0.0140,0.080,PCNO(R),349274.001420
...,...,...,...,...,...,...,...,...,...,...,...
116552,QCOM,QCW2,D463,810674,2026-03-31,0.000,0.147826,0.0000,0.000,PA_ESS_HO,12860.631072
116553,QCOM,QCW2,D463,810971,2026-01-31,0.000,0.454555,0.0000,0.000,PA_ESS_HO,12860.631072
116554,QCOM,QCW2,D463,810971,2026-02-28,0.000,0.454555,0.0000,0.000,PA_ESS_HO,12860.631072
116555,QCOM,QCW2,D463,810971,2026-03-31,0.000,0.434782,0.0000,0.000,PA_ESS_HO,12860.631072


In [38]:
df.isna().sum()

Channel                   0
ASM                       0
Depot                     0
PSKU                      0
month_date                0
pri_actuals_vol_rum       0
pri_apo_plan_vol_rum      0
Consensus Vol             0
Actuals Vol               0
brand                     0
Index Rate              277
dtype: int64

In [39]:
df['Consensus Vol'] = df['Consensus Vol'].fillna(0)
df['Actuals Vol'] = df['Actuals Vol'].fillna(0)

In [244]:
#df['Stat Val'] = df['Stat Vol'] * df['Index Rate'] / (10 ** 7)
df['Consensus Val'] = df['Consensus Vol'] * df['Index Rate'] / (10 ** 7)
df['Actuals Val'] = df['Actuals Vol'] * df['Index Rate'] / (10 ** 7)

In [245]:
df

,Channel,ASM,Depot,PSKU,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,Consensus Vol,Actuals Vol,brand,Index Rate,Consensus Val,Actuals Val
0,B2B,BCE1,D231,718303,2026-01-31,0.000,0.007425,0.0100,0.000,REV.LQDST,247926.608903,0.000248,0.000000
1,B2B,BCE1,D231,718303,2026-02-28,0.000,0.004600,0.0070,0.000,REV.LQDST,247926.608903,0.000174,0.000000
2,B2B,BCE1,D231,718303,2026-03-31,0.014,0.005325,0.0071,0.014,REV.LQDST,247926.608903,0.000176,0.000347
3,B2B,BCE1,D231,718312,2026-01-31,0.040,0.007425,0.0100,0.040,PCNO(R),349274.001420,0.000349,0.001397
4,B2B,BCE1,D231,718312,2026-02-28,0.080,0.009400,0.0140,0.080,PCNO(R),349274.001420,0.000489,0.002794
...,...,...,...,...,...,...,...,...,...,...,...,...,...
116552,QCOM,QCW2,D463,810674,2026-03-31,0.000,0.147826,0.0000,0.000,PA_ESS_HO,12860.631072,0.000000,0.000000
116553,QCOM,QCW2,D463,810971,2026-01-31,0.000,0.454555,0.0000,0.000,PA_ESS_HO,12860.631072,0.000000,0.000000
116554,QCOM,QCW2,D463,810971,2026-02-28,0.000,0.454555,0.0000,0.000,PA_ESS_HO,12860.631072,0.000000,0.000000
116555,QCOM,QCW2,D463,810971,2026-03-31,0.000,0.434782,0.0000,0.000,PA_ESS_HO,12860.631072,0.000000,0.000000


In [246]:
#df['Stat Error'] = df['Stat Val'] - df['Actuals Val']
df['Consensus Error'] = df['Consensus Val'] - df['Actuals Val']

#df['Stat Abs Error'] = np.abs(df['Stat Error'])
df['Consensus Abs Error'] = np.abs(df['Consensus Error'])

## DRM

In [40]:
delivery_df = pd.read_excel('/data/aman_singh/acuuracy_check/Feb Plans.xlsx', sheet_name = 'DRM')

delivery_df['Channel'] = delivery_df['Channel'].replace({
    'E-Commerce': 'ECOM', 'Q-Commerce': 'QCOM'})
delivery_df

,Portfolio,Brand,PSKU,PSKU Desc,Cluster,ASM,Channel,Depot,Feb-26 Del,Feb-26 APO
0,CNO,PCNO(R),718488,PCNO 175ml JAR,CW1,RAJ2,GT,D314,1.7577,1.89
1,Saffola Oils,SAFF GOLD,721898,SAFF GOLD 1L PCH RC,CE2,BIHW,GT,D233,151.1376,164.28
2,Hair Oils,PADVJAS-R,718670,PAR ADV JASMINE 24ml BTL,CS1,VIJ,GT,D572,0.0000,0.00
3,Hair Oils,PADVJAS-R,718793,PAR ADV JASMINE (TR) 400ml BTL,CN2,BCN2,B2B,D115,0.0000,0.00
4,Foods,SFOATS-FL,718559,SMO CLS MSL 38g PCH,CW1,MCW1,MT,D314,0.6900,0.69
...,...,...,...,...,...,...,...,...,...,...
121358,Hair Oils,PA_EN_ML,733326,PA ENRICH ONION 200ML,CN2,HAR,GT,D115,0.0000,0.00
121359,Hair Oils,PA_EN_ML,733326,PA ENRICH ONION 200ML,CN2,HAR,GT,D117,0.0000,0.00
121360,Hair Oils,PA_EN_ML,733326,PA ENRICH ONION 200ML,CW2,AURG,GT,D463,0.0000,0.00
121361,Hair Oils,PA_EN_ML,733326,PA ENRICH ONION 200ML,CE2,BIHE,GT,D232,0.0000,0.00


In [41]:
delivery_df['month'] = '2026-02-28'
delivery_df['delivery_vol'] = delivery_df['Feb-26 Del']

In [42]:
delivery_df_mar = pd.read_excel('/data/aman_singh/acuuracy_check/Mar Plans.xlsx', sheet_name = 'DRM')

delivery_df_mar['Channel'] = delivery_df_mar['Channel'].replace({
    'E-Commerce': 'ECOM', 'Q-Commerce': 'QCOM'})
delivery_df_mar

,Portfolio,Brand,PSKU,PSKU Desc,Cluster,ASM,Channel,Depot,Mar-26 Del,Mar-26 APO
0,CNO,PCNO(R),718488,PCNO 175ml JAR,CW1,RAJ2,GT,D314,1.000000,1.075269
1,Saffola Oils,SAFF GOLD,721898,SAFF GOLD 1L PCH RC,CE2,BIHW,GT,D233,96.780291,96.780291
2,Hair Oils,PADVJAS-R,718670,PAR ADV JASMINE 24ml BTL,CS1,VIJ,GT,D572,0.000000,0.000000
3,Hair Oils,PADVJAS-R,718793,PAR ADV JASMINE (TR) 400ml BTL,CN2,BCN2,B2B,D115,0.000000,0.000000
4,Foods,SFOATS-FL,718559,SMO CLS MSL 38g PCH,CW1,MCW1,MT,D314,0.687768,0.687768
...,...,...,...,...,...,...,...,...,...,...
126259,Hair Oils,H&C_ALMND,732837,HNC ALMOND OIL 100ML+48ML P,CE2,MCE2,MT,D465,84.000000,84.000000
126260,Hair Oils,H&C_ALMND,732837,HNC ALMOND OIL 100ML+48ML P,CE2,MCE2,MT,D234,561.600000,561.600000
126261,Hair Oils,H&C_ALMND,732837,HNC ALMOND OIL 100ML+48ML P,CE2,MCE2,MT,D535,784.000000,784.000000
126262,Hair Oils,H&C_ALMND,732837,HNC ALMOND OIL 100ML+48ML P,CE2,MCE2,MT,D463,67.500000,67.500000


In [43]:
delivery_df_mar['month'] = '2026-03-31'
delivery_df_mar['delivery_vol'] = delivery_df_mar['Mar-26 Del']

In [44]:
delivery_df = pd.concat([delivery_df, delivery_df_mar], ignore_index=True)
delivery_df

,Portfolio,Brand,PSKU,PSKU Desc,Cluster,ASM,Channel,Depot,Feb-26 Del,Feb-26 APO,month,delivery_vol,Mar-26 Del,Mar-26 APO
0,CNO,PCNO(R),718488,PCNO 175ml JAR,CW1,RAJ2,GT,D314,1.7577,1.89,2026-02-28,1.7577,NaN,NaN
1,Saffola Oils,SAFF GOLD,721898,SAFF GOLD 1L PCH RC,CE2,BIHW,GT,D233,151.1376,164.28,2026-02-28,151.1376,NaN,NaN
2,Hair Oils,PADVJAS-R,718670,PAR ADV JASMINE 24ml BTL,CS1,VIJ,GT,D572,0.0000,0.00,2026-02-28,0.0000,NaN,NaN
3,Hair Oils,PADVJAS-R,718793,PAR ADV JASMINE (TR) 400ml BTL,CN2,BCN2,B2B,D115,0.0000,0.00,2026-02-28,0.0000,NaN,NaN
4,Foods,SFOATS-FL,718559,SMO CLS MSL 38g PCH,CW1,MCW1,MT,D314,0.6900,0.69,2026-02-28,0.6900,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
247622,Hair Oils,H&C_ALMND,732837,HNC ALMOND OIL 100ML+48ML P,CE2,MCE2,MT,D465,NaN,NaN,2026-03-31,84.0000,84.0,84.0
247623,Hair Oils,H&C_ALMND,732837,HNC ALMOND OIL 100ML+48ML P,CE2,MCE2,MT,D234,NaN,NaN,2026-03-31,561.6000,561.6,561.6
247624,Hair Oils,H&C_ALMND,732837,HNC ALMOND OIL 100ML+48ML P,CE2,MCE2,MT,D535,NaN,NaN,2026-03-31,784.0000,784.0,784.0
247625,Hair Oils,H&C_ALMND,732837,HNC ALMOND OIL 100ML+48ML P,CE2,MCE2,MT,D463,NaN,NaN,2026-03-31,67.5000,67.5,67.5


In [ ]:
# delivery_df['delivery_vol'] = delivery_df['01-12-2025 Del Vol'].map(lambda x:0 if x.strip() == '-' else float(x.strip().replace(',','')))

In [45]:
# delivery_df.to_csv('delivery_data.csv')
delivery_df.columns = delivery_df.columns.str.lower()

In [46]:
delivery_df = delivery_df.groupby(['channel', 'depot','asm', 'psku','month'])['delivery_vol'].sum().reset_index(
)#.rename(columns = {'01-12-2025 Del Vol':'delivery_vol'})
delivery_df

,channel,depot,asm,psku,month,delivery_vol
0,B2B,D112,BCN1,702478,2026-02-28,0.0
1,B2B,D112,BCN1,702478,2026-03-31,0.0
2,B2B,D112,BCN1,715096,2026-02-28,0.0
3,B2B,D112,BCN1,715096,2026-03-31,0.0
4,B2B,D112,BCN1,715100,2026-02-28,0.0
...,...,...,...,...,...,...
246784,QCOM,D677,QCS2,811069,2026-02-28,0.0
246785,QCOM,D677,QCS2,811069,2026-03-31,0.0
246786,QCOM,D677,QCS2,811149,2026-03-31,0.0
246787,QCOM,D677,QCS2,811169,2026-02-28,0.0


In [47]:
delivery_df['channel'].unique()

array(['B2B', 'CSD', 'D2C', 'ECOM', 'GT', 'MT', 'QCOM'], dtype=object)

In [48]:
delivery_df['delivery_vol'].sum()

9737175.278693762

In [49]:
delivery_df['month_date'] = pd.to_datetime(delivery_df['month'])
delivery_df.drop('month', axis=1, inplace=True)

In [50]:
df.columns = df.columns.str.lower()
df

,channel,asm,depot,psku,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,consensus vol,actuals vol,brand,index rate
0,B2B,BCE1,D231,718303,2026-01-31,0.000,0.007425,0.0100,0.000,REV.LQDST,247926.608903
1,B2B,BCE1,D231,718303,2026-02-28,0.000,0.004600,0.0070,0.000,REV.LQDST,247926.608903
2,B2B,BCE1,D231,718303,2026-03-31,0.014,0.005325,0.0071,0.014,REV.LQDST,247926.608903
3,B2B,BCE1,D231,718312,2026-01-31,0.040,0.007425,0.0100,0.040,PCNO(R),349274.001420
4,B2B,BCE1,D231,718312,2026-02-28,0.080,0.009400,0.0140,0.080,PCNO(R),349274.001420
...,...,...,...,...,...,...,...,...,...,...,...
116552,QCOM,QCW2,D463,810674,2026-03-31,0.000,0.147826,0.0000,0.000,PA_ESS_HO,12860.631072
116553,QCOM,QCW2,D463,810971,2026-01-31,0.000,0.454555,0.0000,0.000,PA_ESS_HO,12860.631072
116554,QCOM,QCW2,D463,810971,2026-02-28,0.000,0.454555,0.0000,0.000,PA_ESS_HO,12860.631072
116555,QCOM,QCW2,D463,810971,2026-03-31,0.000,0.434782,0.0000,0.000,PA_ESS_HO,12860.631072


In [268]:
len_before_merge = len(df)
df = df.merge(
    delivery_df,
    on=['channel', 'asm', 'depot', 'psku','month_date'],
    how='left'
)
assert len_before_merge == len(df)
del len_before_merge

In [269]:
df.rename(columns={'delivery_vol': 'delivery_vol_drm'}, inplace=True)

In [270]:
df

,channel,asm,depot,psku,month_date,pri_actuals_vol_rum,pri_apo_plan_vol_rum,consensus vol,actuals vol,brand,index rate,consensus val,actuals val,consensus error,consensus abs error,delivery_vol_drm
0,B2B,BCE1,D231,718303,2026-01-31,0.000,0.007425,0.0100,0.000,REV.LQDST,247926.608903,0.000248,0.000000,0.000248,0.000248,NaN
1,B2B,BCE1,D231,718303,2026-02-28,0.000,0.004600,0.0070,0.000,REV.LQDST,247926.608903,0.000174,0.000000,0.000174,0.000174,0.010000
2,B2B,BCE1,D231,718303,2026-03-31,0.014,0.005325,0.0071,0.014,REV.LQDST,247926.608903,0.000176,0.000347,-0.000171,0.000171,0.007076
3,B2B,BCE1,D231,718312,2026-01-31,0.040,0.007425,0.0100,0.040,PCNO(R),349274.001420,0.000349,0.001397,-0.001048,0.001048,NaN
4,B2B,BCE1,D231,718312,2026-02-28,0.080,0.009400,0.0140,0.080,PCNO(R),349274.001420,0.000489,0.002794,-0.002305,0.002305,0.010000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
116552,QCOM,QCW2,D463,810674,2026-03-31,0.000,0.147826,0.0000,0.000,PA_ESS_HO,12860.631072,0.000000,0.000000,0.000000,0.000000,0.000000
116553,QCOM,QCW2,D463,810971,2026-01-31,0.000,0.454555,0.0000,0.000,PA_ESS_HO,12860.631072,0.000000,0.000000,0.000000,0.000000,NaN
116554,QCOM,QCW2,D463,810971,2026-02-28,0.000,0.454555,0.0000,0.000,PA_ESS_HO,12860.631072,0.000000,0.000000,0.000000,0.000000,0.000000
116555,QCOM,QCW2,D463,810971,2026-03-31,0.000,0.434782,0.0000,0.000,PA_ESS_HO,12860.631072,0.000000,0.000000,0.000000,0.000000,0.000000


In [271]:
df['delivery val_drm'] = df['delivery_vol_drm'] * df['index rate'] / (10 ** 7)

df['dp error drm'] = df['delivery val_drm'] - df['actuals val']
# df['Consensus Error_del'] = df['Consensus Val'] - df['Delivery Val']

df['dp abs error drm'] = np.abs(df['dp error drm'])
# df['Consensus Abs Error_del'] = np.abs(df['Consensus Error_del'])

In [281]:
df[(df['channel'] == 'GT') & (df['month_date']=='2026-02-28')]['actuals val'].sum()

431.5688636149166

In [285]:
df = df[df['month_date']>='2026-02-28']

In [ ]:
# df['key'] = df['depot'] + '_' + df['psku'].astype(str)
# delivery_df['key'] = delivery_df['depot'] + '_' + delivery_df['psku'].astype(str)


In [296]:
df['actuals vol'].sum()

10813753.241

In [286]:
df.shape

(74798, 19)

In [295]:
df[(df['delivery val_drm'].isna()) & (df['channel']=='GT')].sort_values(by = ['actuals val'], ascending = False).to_csv('missing_keys_drm.csv')#['actuals val'].sum()

In [282]:
df[(df['channel'] == 'GT') & (df['month_date']=='2026-02-28')]['delivery val_drm'].sum()

325.12836666610485

In [57]:
df.isnull().sum()

Channel                 0
Portfolio               0
Brand                   0
Key                     0
Month                   0
ASM                     0
Depot                   0
PSKU                    0
Stat Vol                0
Consensus Vol           0
Actuals Vol             0
Index Rate              0
Stat Val                0
Consensus Val           0
Actuals Val             0
Stat Error              0
Consensus Error         0
Stat Abs Error          0
Consensus Abs Error     0
delivery_vol           37
Delivery Val           37
Dp Error               37
Dp Abs Error           37
dtype: int64

In [58]:
df.to_excel("ADP Accuracy DEC'25_2.xlsx", index=False)

### depot psku level

In [51]:
delivery_df = delivery_df.groupby(['channel','depot', 'psku','month_date'])['delivery_vol'].sum().reset_index()
delivery_df

,channel,depot,psku,month_date,delivery_vol
0,B2B,D112,702478,2026-02-28,0.0
1,B2B,D112,702478,2026-03-31,0.0
2,B2B,D112,715096,2026-02-28,0.0
3,B2B,D112,715096,2026-03-31,0.0
4,B2B,D112,715100,2026-02-28,0.0
...,...,...,...,...,...
200815,QCOM,D677,811069,2026-02-28,0.0
200816,QCOM,D677,811069,2026-03-31,0.0
200817,QCOM,D677,811149,2026-03-31,0.0
200818,QCOM,D677,811169,2026-02-28,0.0


In [52]:
df.columns = df.columns.str.lower()

In [53]:
df = df.groupby(['channel','depot', 'psku','month_date', 'brand'])[['consensus vol', 'actuals vol']].sum().reset_index()
df

,channel,depot,psku,month_date,brand,consensus vol,actuals vol
0,B2B,BUDZ,718303,2026-03-31,REV.LQDST,0.0,0.0
1,B2B,BUDZ,718312,2026-03-31,PCNO(R),0.0,0.0
2,B2B,BUDZ,718317,2026-03-31,H&C,0.0,0.0
3,B2B,BUDZ,718318,2026-03-31,H&C,0.0,0.0
4,B2B,BUDZ,718325,2026-03-31,REV.ST.,0.0,0.0
...,...,...,...,...,...,...,...
78839,QCOM,D677,810971,2026-02-28,PA_ESS_HO,0.0,0.0
78840,QCOM,D677,810971,2026-03-31,PA_ESS_HO,0.0,0.0
78841,QCOM,D677,811169,2026-03-31,SW_SGPRF,0.0,0.0
78842,QCOM,D677,811181,2026-02-28,SAF_CDPRS,0.0,0.0


In [54]:
len_before_merge = len(df)
df = df.merge(
    qtr_ind_rate_df.drop('month_date', axis=1).rename(
        columns={'brand_code': 'brand', 'qtr_ind_rate': 'Index Rate'}
    ),
    on=['brand'], 
    how='left'
)
assert len_before_merge == len(df)
del len_before_merge



In [55]:
df['consensus vol'] = df['consensus vol'].fillna(0)
df['actuals vol'] = df['actuals vol'].fillna(0)
#df['Stat Val'] = df['Stat Vol'] * df['Index Rate'] / (10 ** 7)
df['Consensus Val'] = df['consensus vol'] * df['Index Rate'] / (10 ** 7)
df['Actuals Val'] = df['actuals vol'] * df['Index Rate'] / (10 ** 7)

#df['Stat Error'] = df['Stat Val'] - df['Actuals Val']
df['Consensus Error'] = df['Consensus Val'] - df['Actuals Val']

#df['Stat Abs Error'] = np.abs(df['Stat Error'])
df['Consensus Abs Error'] = np.abs(df['Consensus Error'])

In [413]:
df

,channel,depot,psku,month_date,brand,consensus vol,actuals vol,Index Rate,Consensus Val,Actuals Val,Consensus Error,Consensus Abs Error
0,B2B,BUDZ,718303,2026-03-31,REV.LQDST,0.0,0.0,247926.608903,0.0,0.0,0.0,0.0
1,B2B,BUDZ,718312,2026-03-31,PCNO(R),0.0,0.0,349274.001420,0.0,0.0,0.0,0.0
2,B2B,BUDZ,718317,2026-03-31,H&C,0.0,0.0,388.076436,0.0,0.0,0.0,0.0
3,B2B,BUDZ,718318,2026-03-31,H&C,0.0,0.0,388.076436,0.0,0.0,0.0,0.0
4,B2B,BUDZ,718325,2026-03-31,REV.ST.,0.0,0.0,253390.161725,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
78839,QCOM,D677,810971,2026-02-28,PA_ESS_HO,0.0,0.0,12860.631072,0.0,0.0,0.0,0.0
78840,QCOM,D677,810971,2026-03-31,PA_ESS_HO,0.0,0.0,12860.631072,0.0,0.0,0.0,0.0
78841,QCOM,D677,811169,2026-03-31,SW_SGPRF,0.0,0.0,1712.605337,0.0,0.0,0.0,0.0
78842,QCOM,D677,811181,2026-02-28,SAF_CDPRS,0.0,0.0,260000.000000,0.0,0.0,0.0,0.0


In [56]:
df = df.merge(delivery_df, on=['channel', 'depot', 'psku','month_date'], how='left')
df

,channel,depot,psku,month_date,brand,consensus vol,actuals vol,Index Rate,Consensus Val,Actuals Val,Consensus Error,Consensus Abs Error,delivery_vol
0,B2B,BUDZ,718303,2026-03-31,REV.LQDST,0.0,0.0,247926.608903,0.0,0.0,0.0,0.0,NaN
1,B2B,BUDZ,718312,2026-03-31,PCNO(R),0.0,0.0,349274.001420,0.0,0.0,0.0,0.0,NaN
2,B2B,BUDZ,718317,2026-03-31,H&C,0.0,0.0,388.076436,0.0,0.0,0.0,0.0,NaN
3,B2B,BUDZ,718318,2026-03-31,H&C,0.0,0.0,388.076436,0.0,0.0,0.0,0.0,NaN
4,B2B,BUDZ,718325,2026-03-31,REV.ST.,0.0,0.0,253390.161725,0.0,0.0,0.0,0.0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
78839,QCOM,D677,810971,2026-02-28,PA_ESS_HO,0.0,0.0,12860.631072,0.0,0.0,0.0,0.0,0.0
78840,QCOM,D677,810971,2026-03-31,PA_ESS_HO,0.0,0.0,12860.631072,0.0,0.0,0.0,0.0,0.0
78841,QCOM,D677,811169,2026-03-31,SW_SGPRF,0.0,0.0,1712.605337,0.0,0.0,0.0,0.0,0.0
78842,QCOM,D677,811181,2026-02-28,SAF_CDPRS,0.0,0.0,260000.000000,0.0,0.0,0.0,0.0,NaN


In [57]:
df.rename(columns={'delivery_vol': 'delivery_vol_drm'}, inplace=True)

In [58]:
df['delivery_vol_drm'].sum(), df['actuals vol'].sum()

(9198620.245054696, 16267037.818)

In [59]:
df.columns = df.columns.str.lower()

In [60]:
df['delivery val_drm'] = df['delivery_vol_drm'] * df['index rate'] / (10 ** 7)

df['dp error drm'] = df['delivery val_drm'] - df['actuals val']
# df['Consensus Error_del'] = df['Consensus Val'] - df['Delivery Val']

df['dp abs error drm'] = np.abs(df['dp error drm'])
# df['Consensus Abs Error_del'] = np.abs(df['Consensus Error_del'])

In [344]:

df[df['delivery_vol_drm'].isna() ].sort_values(by = ['actuals val'], ascending = False).to_csv('missing_keys_drm2.csv')#['actuals val'].sum()

In [61]:
df[(df['channel'] == 'GT') & (df['month_date']=='2026-03-31')]['actuals val'].sum()

330.5686096579248

In [62]:
df = df[df['month_date']>='2026-02-28']
df

,channel,depot,psku,month_date,brand,consensus vol,actuals vol,index rate,consensus val,actuals val,consensus error,consensus abs error,delivery_vol_drm,delivery val_drm,dp error drm,dp abs error drm
0,B2B,BUDZ,718303,2026-03-31,REV.LQDST,0.0,0.0,247926.608903,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN
1,B2B,BUDZ,718312,2026-03-31,PCNO(R),0.0,0.0,349274.001420,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN
2,B2B,BUDZ,718317,2026-03-31,H&C,0.0,0.0,388.076436,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN
3,B2B,BUDZ,718318,2026-03-31,H&C,0.0,0.0,388.076436,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN
4,B2B,BUDZ,718325,2026-03-31,REV.ST.,0.0,0.0,253390.161725,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
78839,QCOM,D677,810971,2026-02-28,PA_ESS_HO,0.0,0.0,12860.631072,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
78840,QCOM,D677,810971,2026-03-31,PA_ESS_HO,0.0,0.0,12860.631072,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
78841,QCOM,D677,811169,2026-03-31,SW_SGPRF,0.0,0.0,1712.605337,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
78842,QCOM,D677,811181,2026-02-28,SAF_CDPRS,0.0,0.0,260000.000000,0.0,0.0,0.0,0.0,NaN,NaN,NaN,NaN


In [ ]:
# df.to_csv('fva_depot_psku.csv', index=False)

In [6]:
df = pd.read_excel('/data/aman_singh/acuuracy_check/acc_framework_apr_final.xlsx')
df

,Unnamed: 0,channel,portfolio,brand,run_month,m month,month,depot,psku,stat vol,...,stat abs error,consensus abs error,prophet vol_value,rf_vol_value,prophet heuristic vol_value,rf heuristic vol_value,stat_bias,stat_bias_bucket,forecast_granularity,forecast_type
0,122014,GT,CNO,KERALA,2026-03-31,M+1,2026-04-30,D673,718314,10.786110,...,0.056469,0.091702,0.0,0.0,0.0,0.0,0.429760,> 15%,Depot x PSKU,secondary
1,122015,GT,CNO,KERALA,2026-03-31,M+1,2026-04-30,D673,718339,1.438602,...,0.000759,0.002296,0.0,0.0,0.0,0.0,0.031256,0% to 5%,Depot x PSKU,secondary
2,122016,GT,CNO,KERALA,2026-03-31,M+1,2026-04-30,D674,718314,0.788796,...,0.004967,0.011196,0.0,0.0,0.0,0.0,-0.265553,< -15%,Depot x PSKU,secondary
3,122017,GT,CNO,KERALA,2026-03-31,M+1,2026-04-30,D674,718339,4.326253,...,0.029414,0.017677,0.0,0.0,0.0,0.0,-0.280756,< -15%,Depot x PSKU,secondary
4,122018,GT,CNO,KERALA,2026-03-31,M+1,2026-04-30,D676,718314,40.728186,...,0.421625,0.069203,0.0,0.0,0.0,0.0,1.465237,> 15%,Depot x PSKU,secondary
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29615,320014,Ecom,Skin Care,PABABY_GM,2026-03-31,M+1,2026-04-30,D677,810805,0.000000,...,0.000000,0.000000,NaN,NaN,NaN,NaN,0.000000,0% to 5%,Depot x PSKU,offtakes_to_primary
29616,320015,Ecom,Skin Care,PABABY_GM,2026-03-31,M+1,2026-04-30,D677,810807,0.000000,...,0.000000,0.000000,NaN,NaN,NaN,NaN,0.000000,0% to 5%,Depot x PSKU,offtakes_to_primary
29617,320016,Ecom,Skin Care,PABABY_GM,2026-03-31,M+1,2026-04-30,D677,810919,0.000000,...,0.000000,0.000000,NaN,NaN,NaN,NaN,0.000000,0% to 5%,Depot x PSKU,offtakes_to_primary
29618,320017,Ecom,Male Grooming,SW_SGPRF,2026-03-31,M+1,2026-04-30,D677,811169,0.000000,...,0.000000,0.000000,NaN,NaN,NaN,NaN,0.000000,0% to 5%,Depot x PSKU,offtakes_to_primary


In [7]:
df.rename(columns = {'month':'month_date'}, inplace = True)
df.drop(columns = ['index rate'], inplace = True)

In [8]:
def read_qtr_ind_rate_table():
    query = """select * from DWH_SAP_INDEX_TURNOVER_MONTHWISE 
                where latest_rate_flag=1 and company_code='MIL'"""
    qtr_ind_rate_data = pd.read_sql(con=prod_conn, sql=query)
    qtr_ind_rate_data.columns = qtr_ind_rate_data.columns.str.lower()
    qtr_ind_rate =  qtr_ind_rate_data[['date', 'brand_code', 'turnover']]
    qtr_ind_rate = qtr_ind_rate.rename(columns= {'date':'month_date', 'turnover':'qtr_ind_rate'})
    
    return qtr_ind_rate

qtr_ind_rate_df = read_qtr_ind_rate_table()
qtr_ind_rate_df.head()

len_before_merge = len(df)
df = df.merge(
    qtr_ind_rate_df.drop('month_date', axis=1).rename(
        columns={'brand_code': 'brand', 'qtr_ind_rate': 'index rate'}
    ),
    on=['brand'], 
    how='left'
)
assert len_before_merge == len(df)
del len_before_merge

In [9]:
df['channel'] = df['channel'].str.upper()

### DRM

In [10]:
delivery_df = pd.read_excel('/data/aman_singh/acuuracy_check/April-26 Plans.xlsx', sheet_name = 'DRM')

delivery_df['Channel'] = delivery_df['Channel'].replace({
    'E-Commerce': 'ECOM', 'Q-Commerce': 'QCOM'})
delivery_df['month'] = '2026-04-30'
delivery_df['delivery_vol'] = delivery_df['APR-26 DRM']
# delivery_df_mar = pd.read_excel('/data/aman_singh/acuuracy_check/Mar Plans.xlsx', sheet_name = 'BAM')

# delivery_df_mar['Channel'] = delivery_df_mar['Channel'].replace({
#     'E-Commerce': 'ECOM', 'Q-Commerce': 'QCOM'})
# delivery_df_mar['month'] = '2026-03-31'
# delivery_df_mar['delivery_vol'] = delivery_df_mar['Mar-26 Del']
# delivery_df = pd.concat([delivery_df, delivery_df_mar], ignore_index=True)
# delivery_df['delivery_vol'] = delivery_df['01-12-2025 Del Vol'].map(lambda x:0 if x.strip() == '-' else float(x.strip().replace(',','')))
# delivery_df.to_csv('delivery_data.csv')
delivery_df.columns = delivery_df.columns.str.lower()
delivery_df = delivery_df.groupby(['channel', 'depot','asm', 'psku','month'])['delivery_vol'].sum().reset_index(
)#.rename(columns = {'01-12-2025 Del Vol':'delivery_vol'})
delivery_df['month_date'] = pd.to_datetime(delivery_df['month'])
delivery_df.drop('month', axis=1, inplace=True)
delivery_df

,channel,depot,asm,psku,delivery_vol,month_date
0,B2B,D112,BCN1,718287,0.000000,2026-04-30
1,B2B,D112,BCN1,718288,0.000000,2026-04-30
2,B2B,D112,BCN1,718289,0.000000,2026-04-30
3,B2B,D112,BCN1,718297,0.000000,2026-04-30
4,B2B,D112,BCN1,718299,0.000000,2026-04-30
...,...,...,...,...,...,...
75372,QCOM,D677,QCS2,811149,0.000000,2026-04-30
75373,QCOM,D677,QCS2,811169,0.000000,2026-04-30
75374,QCOM,D677,QCS2,811267,0.237097,2026-04-30
75375,QCOM,D677,QCS2,811279,0.000000,2026-04-30


In [11]:
delivery_df = delivery_df.groupby(['channel','depot', 'psku','month_date'])['delivery_vol'].sum().reset_index()
delivery_df

,channel,depot,psku,month_date,delivery_vol
0,B2B,D112,718287,2026-04-30,0.000000
1,B2B,D112,718288,2026-04-30,0.000000
2,B2B,D112,718289,2026-04-30,0.000000
3,B2B,D112,718297,2026-04-30,0.000000
4,B2B,D112,718299,2026-04-30,0.000000
...,...,...,...,...,...
59546,QCOM,D677,811149,2026-04-30,0.000000
59547,QCOM,D677,811169,2026-04-30,0.000000
59548,QCOM,D677,811267,2026-04-30,0.237097
59549,QCOM,D677,811279,2026-04-30,0.000000


In [12]:
delivery_df['channel'].unique()

array(['B2B', 'CSD', 'ECOM', 'GT', 'MT', 'QCOM'], dtype=object)

In [13]:
df.shape

(29620, 33)

In [14]:
df.columns

Index(['Unnamed: 0', 'channel', 'portfolio', 'brand', 'run_month', 'm month',
       'month_date', 'depot', 'psku', 'stat vol', 'prophet vol', 'rf_vol',
       'prophet heuristic vol', 'rf heuristic vol', 'consensus vol',
       'actuals vol', 'stat val', 'consensus val', 'actuals val', 'key',
       'stat error', 'consensus error', 'stat abs error',
       'consensus abs error', 'prophet vol_value', 'rf_vol_value',
       'prophet heuristic vol_value', 'rf heuristic vol_value', 'stat_bias',
       'stat_bias_bucket', 'forecast_granularity', 'forecast_type',
       'index rate'],
      dtype='object')

In [15]:
df = df.merge(delivery_df, on=['channel', 'depot', 'psku','month_date'], how='left')
df.rename(columns={'delivery_vol': 'delivery_vol_drm'}, inplace=True)
df['delivery_vol_drm'].sum(), df['actuals vol'].sum()
df.columns = df.columns.str.lower()
df['delivery val_drm'] = df['delivery_vol_drm'] * df['index rate'] / (10 ** 7)

df['dp error drm'] = df['delivery val_drm'] - df['actuals val']
# df['Consensus Error_del'] = df['Consensus Val'] - df['Delivery Val']

df['dp abs error drm'] = np.abs(df['dp error drm'])
# df['Consensus Abs Error_del'] = np.abs(df['Consensus Error_del'])



In [16]:
df.isnull().sum()

unnamed: 0                         0
channel                            0
portfolio                          0
brand                              0
run_month                          0
m month                            0
month_date                         0
depot                              0
psku                               0
stat vol                           0
prophet vol                    20064
rf_vol                         20064
prophet heuristic vol          20064
rf heuristic vol               20064
consensus vol                      0
actuals vol                        0
stat val                           0
consensus val                      0
actuals val                        0
key                            20064
stat error                         0
consensus error                    0
stat abs error                     0
consensus abs error                0
prophet vol_value              20064
rf_vol_value                   20064
prophet heuristic vol_value    20064
r

In [17]:
df[(df['channel'] == 'GT') & (df['month_date']=='2026-04-30')]['delivery val_drm'].sum()

487.38678620217377

### CAM

In [18]:
delivery_df = pd.read_excel('/data/aman_singh/acuuracy_check/April-26 Plans.xlsx', sheet_name = 'CAM')

delivery_df['Channel'] = delivery_df['Channel'].replace({
    'E-Commerce': 'ECOM', 'Q-Commerce': 'QCOM'})
delivery_df['month'] = '2026-04-30'
delivery_df['delivery_vol'] = delivery_df['APR-26 CAM']
# delivery_df_mar = pd.read_excel('/data/aman_singh/acuuracy_check/Mar Plans.xlsx', sheet_name = 'CAM')

# delivery_df_mar['Channel'] = delivery_df_mar['Channel'].replace({
#     'E-Commerce': 'ECOM', 'Q-Commerce': 'QCOM'})
# delivery_df_mar['month'] = '2026-03-31'
# delivery_df_mar['delivery_vol'] = delivery_df_mar['Mar-26 Del']
# delivery_df = pd.concat([delivery_df, delivery_df_mar], ignore_index=True)
# delivery_df['delivery_vol'] = delivery_df['01-12-2025 Del Vol'].map(lambda x:0 if x.strip() == '-' else float(x.strip().replace(',','')))
# delivery_df.to_csv('delivery_data.csv')
delivery_df.columns = delivery_df.columns.str.lower()
delivery_df = delivery_df.groupby(['channel', 'depot','asm', 'psku','month'])['delivery_vol'].sum().reset_index(
)#.rename(columns = {'01-12-2025 Del Vol':'delivery_vol'})
delivery_df['month_date'] = pd.to_datetime(delivery_df['month'])
delivery_df.drop('month', axis=1, inplace=True)
delivery_df

,channel,depot,asm,psku,delivery_vol,month_date
0,B2B,D112,BCN1,718287,0.00,2026-04-30
1,B2B,D112,BCN1,718288,0.00,2026-04-30
2,B2B,D112,BCN1,718289,0.00,2026-04-30
3,B2B,D112,BCN1,718297,0.00,2026-04-30
4,B2B,D112,BCN1,718299,0.00,2026-04-30
...,...,...,...,...,...,...
74981,QCOM,D677,QCS2,811021,3.19,2026-04-30
74982,QCOM,D677,QCS2,811068,0.00,2026-04-30
74983,QCOM,D677,QCS2,811069,0.00,2026-04-30
74984,QCOM,D677,QCS2,811149,0.00,2026-04-30


In [19]:
delivery_df = delivery_df.groupby(['channel','depot', 'psku','month_date'])['delivery_vol'].sum().reset_index()
delivery_df

,channel,depot,psku,month_date,delivery_vol
0,B2B,D112,718287,2026-04-30,0.00
1,B2B,D112,718288,2026-04-30,0.00
2,B2B,D112,718289,2026-04-30,0.00
3,B2B,D112,718297,2026-04-30,0.00
4,B2B,D112,718299,2026-04-30,0.00
...,...,...,...,...,...
59266,QCOM,D677,811021,2026-04-30,3.19
59267,QCOM,D677,811068,2026-04-30,0.00
59268,QCOM,D677,811069,2026-04-30,0.00
59269,QCOM,D677,811149,2026-04-30,0.00


In [20]:
df.shape

(29620, 37)

In [21]:
df = df.merge(delivery_df, on=['channel', 'depot', 'psku','month_date'], how='left')
df.rename(columns={'delivery_vol': 'delivery_vol_cam'}, inplace=True)
df['delivery_vol_cam'].sum(), df['actuals vol'].sum()
df.columns = df.columns.str.lower()
df['delivery val_cam'] = df['delivery_vol_cam'] * df['index rate'] / (10 ** 7)

df['dp error cam'] = df['delivery val_cam'] - df['actuals val']
# df['Consensus Error_del'] = df['Consensus Val'] - df['Delivery Val']

df['dp abs error cam'] = np.abs(df['dp error cam'])
# df['Consensus Abs Error_del'] = np.abs(df['Consensus Error_del'])



In [22]:
df

,unnamed: 0,channel,portfolio,brand,run_month,m month,month_date,depot,psku,stat vol,...,forecast_type,index rate,delivery_vol_drm,delivery val_drm,dp error drm,dp abs error drm,delivery_vol_cam,delivery val_cam,dp error cam,dp abs error cam
0,122014,GT,CNO,KERALA,2026-03-31,M+1,2026-04-30,D673,718314,10.786110,...,secondary,174173.594455,0.837586,0.014589,-0.116808,0.116808,0.827570,0.014414,-0.116982,0.116982
1,122015,GT,CNO,KERALA,2026-03-31,M+1,2026-04-30,D673,718339,1.438602,...,secondary,174173.594455,0.530000,0.009231,-0.015066,0.015066,0.530000,0.009231,-0.015066,0.015066
2,122016,GT,CNO,KERALA,2026-03-31,M+1,2026-04-30,D674,718314,0.788796,...,secondary,174173.594455,0.090000,0.001568,-0.017139,0.017139,0.090000,0.001568,-0.017139,0.017139
3,122017,GT,CNO,KERALA,2026-03-31,M+1,2026-04-30,D674,718339,4.326253,...,secondary,174173.594455,5.590000,0.097363,-0.007402,0.007402,5.590000,0.097363,-0.007402,0.007402
4,122018,GT,CNO,KERALA,2026-03-31,M+1,2026-04-30,D676,718314,40.728186,...,secondary,174173.594455,11.861212,0.206591,-0.081161,0.081161,11.861212,0.206591,-0.081161,0.081161
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29615,320014,ECOM,Skin Care,PABABY_GM,2026-03-31,M+1,2026-04-30,D677,810805,0.000000,...,offtakes_to_primary,366.484998,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
29616,320015,ECOM,Skin Care,PABABY_GM,2026-03-31,M+1,2026-04-30,D677,810807,0.000000,...,offtakes_to_primary,366.484998,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
29617,320016,ECOM,Skin Care,PABABY_GM,2026-03-31,M+1,2026-04-30,D677,810919,0.000000,...,offtakes_to_primary,366.484998,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
29618,320017,ECOM,Male Grooming,SW_SGPRF,2026-03-31,M+1,2026-04-30,D677,811169,0.000000,...,offtakes_to_primary,1712.605337,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [23]:
df[(df['channel'] == 'GT') & (df['month_date']=='2026-04-30')]['delivery val_cam'].sum()

484.7806873195155

### BAM

In [24]:
delivery_df = pd.read_excel('/data/aman_singh/acuuracy_check/April-26 Plans.xlsx', sheet_name = 'BAM')

delivery_df['Channel'] = delivery_df['Channel'].replace({
    'E-Commerce': 'ECOM', 'Q-Commerce': 'QCOM'})
delivery_df['month'] = '2026-04-30'
delivery_df['delivery_vol'] = delivery_df['APR-26 BAM']
# delivery_df_mar = pd.read_excel('/data/aman_singh/acuuracy_check/Mar Plans.xlsx', sheet_name = 'BAM')

# delivery_df_mar['Channel'] = delivery_df_mar['Channel'].replace({
#     'E-Commerce': 'ECOM', 'Q-Commerce': 'QCOM'})
# delivery_df_mar['month'] = '2026-03-31'
# delivery_df_mar['delivery_vol'] = delivery_df_mar['Mar-26 Del']
# delivery_df = pd.concat([delivery_df, delivery_df_mar], ignore_index=True)
# delivery_df['delivery_vol'] = delivery_df['01-12-2025 Del Vol'].map(lambda x:0 if x.strip() == '-' else float(x.strip().replace(',','')))
# delivery_df.to_csv('delivery_data.csv')
delivery_df.columns = delivery_df.columns.str.lower()
delivery_df = delivery_df.groupby(['channel', 'depot','asm', 'psku','month'])['delivery_vol'].sum().reset_index(
)#.rename(columns = {'01-12-2025 Del Vol':'delivery_vol'})
delivery_df['month_date'] = pd.to_datetime(delivery_df['month'])
delivery_df.drop('month', axis=1, inplace=True)
delivery_df

,channel,depot,asm,psku,delivery_vol,month_date
0,B2B,D112,BCN1,718287,0.000000,2026-04-30
1,B2B,D112,BCN1,718288,0.000000,2026-04-30
2,B2B,D112,BCN1,718289,0.000000,2026-04-30
3,B2B,D112,BCN1,718297,0.000000,2026-04-30
4,B2B,D112,BCN1,718299,0.000000,2026-04-30
...,...,...,...,...,...,...
75575,QCOM,D677,QCS2,811149,0.000000,2026-04-30
75576,QCOM,D677,QCS2,811169,0.000000,2026-04-30
75577,QCOM,D677,QCS2,811267,0.237097,2026-04-30
75578,QCOM,D677,QCS2,811279,0.000000,2026-04-30


In [25]:
delivery_df = delivery_df.groupby(['channel','depot', 'psku','month_date'])['delivery_vol'].sum().reset_index()
delivery_df

,channel,depot,psku,month_date,delivery_vol
0,B2B,D112,718287,2026-04-30,0.000000
1,B2B,D112,718288,2026-04-30,0.000000
2,B2B,D112,718289,2026-04-30,0.000000
3,B2B,D112,718297,2026-04-30,0.000000
4,B2B,D112,718299,2026-04-30,0.000000
...,...,...,...,...,...
59667,QCOM,D677,811149,2026-04-30,0.000000
59668,QCOM,D677,811169,2026-04-30,0.000000
59669,QCOM,D677,811267,2026-04-30,0.237097
59670,QCOM,D677,811279,2026-04-30,0.000000


In [26]:
df.shape

(29620, 41)

In [27]:
df = df.merge(delivery_df, on=['channel', 'depot', 'psku','month_date'], how='left')
df.rename(columns={'delivery_vol': 'delivery_vol_bam'}, inplace=True)
df['delivery_vol_bam'].sum(), df['actuals vol'].sum()
df.columns = df.columns.str.lower()
df['delivery val_bam'] = df['delivery_vol_bam'] * df['index rate'] / (10 ** 7)

df['dp error bam'] = df['delivery val_bam'] - df['actuals val']
# df['Consensus Error_del'] = df['Consensus Val'] - df['Delivery Val']

df['dp abs error bam'] = np.abs(df['dp error bam'])
# df['Consensus Abs Error_del'] = np.abs(df['Consensus Error_del'])



In [28]:
df

,unnamed: 0,channel,portfolio,brand,run_month,m month,month_date,depot,psku,stat vol,...,dp error drm,dp abs error drm,delivery_vol_cam,delivery val_cam,dp error cam,dp abs error cam,delivery_vol_bam,delivery val_bam,dp error bam,dp abs error bam
0,122014,GT,CNO,KERALA,2026-03-31,M+1,2026-04-30,D673,718314,10.786110,...,-0.116808,0.116808,0.827570,0.014414,-0.116982,0.116982,0.778638,0.013562,-0.117835,0.117835
1,122015,GT,CNO,KERALA,2026-03-31,M+1,2026-04-30,D673,718339,1.438602,...,-0.015066,0.015066,0.530000,0.009231,-0.015066,0.015066,0.500000,0.008709,-0.015589,0.015589
2,122016,GT,CNO,KERALA,2026-03-31,M+1,2026-04-30,D674,718314,0.788796,...,-0.017139,0.017139,0.090000,0.001568,-0.017139,0.017139,0.100000,0.001742,-0.016965,0.016965
3,122017,GT,CNO,KERALA,2026-03-31,M+1,2026-04-30,D674,718339,4.326253,...,-0.007402,0.007402,5.590000,0.097363,-0.007402,0.007402,5.600000,0.097537,-0.007228,0.007228
4,122018,GT,CNO,KERALA,2026-03-31,M+1,2026-04-30,D676,718314,40.728186,...,-0.081161,0.081161,11.861212,0.206591,-0.081161,0.081161,11.900000,0.207267,-0.080486,0.080486
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29615,320014,ECOM,Skin Care,PABABY_GM,2026-03-31,M+1,2026-04-30,D677,810805,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
29616,320015,ECOM,Skin Care,PABABY_GM,2026-03-31,M+1,2026-04-30,D677,810807,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
29617,320016,ECOM,Skin Care,PABABY_GM,2026-03-31,M+1,2026-04-30,D677,810919,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
29618,320017,ECOM,Male Grooming,SW_SGPRF,2026-03-31,M+1,2026-04-30,D677,811169,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [29]:
df[(df['channel'] == 'GT') & (df['month_date']=='2026-04-30')]['delivery val_bam'].sum()

487.07360194172475

In [52]:
df.to_csv('fva_depot_psku_april.csv', index=False)

In [30]:
previous_data = pd.read_excel('/data/aman_singh/acuuracy_check/fva_and_stat_accuracy.xlsx', sheet_name = 'fva_depot_psku_final')
previous_data

,forecast_granularity,forecast_type,channel,depot,psku,brand,portfolio,month,m month,stat vol,...,bias_bucket_bam,bias_bucket_plan,chain,offtakes_forecasted_value,offtakes_forecasted_vol,offtakes actuals val,offtakes error,offtakes abs error,offtakes_bias,offtakes_bias_bucket
0,Depot x PSKU,secondary,B2B,D112,718287,PCNO(R),CNO,2025-11-30,M+1,0.0,...,5 | 0% to 5%,5 | 0% to 5%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Depot x PSKU,secondary,B2B,D112,718287,PCNO(R),CNO,2025-12-31,M+1,0.0,...,5 | 0% to 5%,5 | 0% to 5%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Depot x PSKU,secondary,B2B,D112,718297,PCNO(R),CNO,2025-11-30,M+1,0.0,...,5 | 0% to 5%,5 | 0% to 5%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Depot x PSKU,secondary,B2B,D112,718297,PCNO(R),CNO,2025-12-31,M+1,0.0,...,5 | 0% to 5%,5 | 0% to 5%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Depot x PSKU,secondary,B2B,D112,718297,PCNO(R),CNO,2026-01-31,M+1,0.0,...,5 | 0% to 5%,5 | 0% to 5%,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
153484,Depot x PSKU,secondary,B2B,D677,808719,SAF_HONEY,NaN,2026-03-31,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
153485,Depot x PSKU,secondary,B2B,D677,808721,SAF_HONEY,NaN,2026-03-31,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
153486,Depot x PSKU,secondary,B2B,D677,809814,SAF-MUSLI,NaN,2026-03-31,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
153487,Depot x PSKU,secondary,B2B,D677,809815,SAF-MUSLI,NaN,2026-03-31,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [31]:
previous_data['month'].unique()

<DatetimeArray>
['2025-11-30 00:00:00', '2025-12-31 00:00:00', '2026-01-31 00:00:00',
 '2026-02-28 00:00:00', '2026-03-31 00:00:00']
Length: 5, dtype: datetime64[ns]

In [32]:
previous_data.columns

Index(['forecast_granularity', 'forecast_type', 'channel', 'depot', 'psku',
       'brand', 'portfolio', 'month', 'm month', 'stat vol', 'actuals vol',
       'consensus vol', 'index rate', 'stat val', 'consensus val',
       'actuals val', 'stat error', 'consensus error', 'stat abs error',
       'consensus abs error', 'stat_bias', 'stat_bias_bucket',
       'delivery_vol_drm', 'delivery val_drm', 'delivery_vol_cam',
       'delivery val_cam', 'delivery_vol_bam', 'delivery val_bam',
       'brand class', 'dp error drm', 'dp abs error drm', 'dp error cam',
       'dp abs error cam', 'dp error bam', 'dp abs error bam',
       'Bias bucket drm', 'bias bucket cam', 'bias_bucket_bam',
       'bias_bucket_plan', 'chain', 'offtakes_forecasted_value',
       'offtakes_forecasted_vol', 'offtakes actuals val', 'offtakes error',
       'offtakes abs error', 'offtakes_bias', 'offtakes_bias_bucket'],
      dtype='object')

In [33]:
df.rename(columns = {'month_date':'month'}, inplace = True)
final_data = pd.concat([previous_data, df])
final_data

,forecast_granularity,forecast_type,channel,depot,psku,brand,portfolio,month,m month,stat vol,...,run_month,prophet vol,rf_vol,prophet heuristic vol,rf heuristic vol,key,prophet vol_value,rf_vol_value,prophet heuristic vol_value,rf heuristic vol_value
0,Depot x PSKU,secondary,B2B,D112,718287,PCNO(R),CNO,2025-11-30,M+1,0.0,...,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Depot x PSKU,secondary,B2B,D112,718287,PCNO(R),CNO,2025-12-31,M+1,0.0,...,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,Depot x PSKU,secondary,B2B,D112,718297,PCNO(R),CNO,2025-11-30,M+1,0.0,...,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Depot x PSKU,secondary,B2B,D112,718297,PCNO(R),CNO,2025-12-31,M+1,0.0,...,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,Depot x PSKU,secondary,B2B,D112,718297,PCNO(R),CNO,2026-01-31,M+1,0.0,...,NaT,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
29615,Depot x PSKU,offtakes_to_primary,ECOM,D677,810805,PABABY_GM,Skin Care,2026-04-30,M+1,0.0,...,2026-03-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
29616,Depot x PSKU,offtakes_to_primary,ECOM,D677,810807,PABABY_GM,Skin Care,2026-04-30,M+1,0.0,...,2026-03-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
29617,Depot x PSKU,offtakes_to_primary,ECOM,D677,810919,PABABY_GM,Skin Care,2026-04-30,M+1,0.0,...,2026-03-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
29618,Depot x PSKU,offtakes_to_primary,ECOM,D677,811169,SW_SGPRF,Male Grooming,2026-04-30,M+1,0.0,...,2026-03-31,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [34]:
final_data.to_csv('stat_fva_till_april2.csv')#['month'].unique()